# FinalCode Surabaya — Adaptasi ML2 untuk 258 PDF Putusan PN Surabaya
Salinan termodifikasi dari `ML2/FinalCode.ipynb` (105 cells) + backbone fitur lengkap `Testing/600crf-pos-prevnext-wordfrom.ipynb`.

**Yang dibuang:** scraping `putusan3.mahkamahagung.go.id` (cell 0-16 asli), `drive.mount()`, training POS Flair 10 epoch (butuh GPU, cell 61-62 asli), `dropna()` Excel, `clean_text v2`, pola Madura `bkl/smp`.

**Yang dipertahankan:** `read_pdf/clean_page/clean_text v1`, `labeling_token/anotasi`, `split_token/fragmentasi`, inferensi POS via `best-model.pt`, CRF `lbfgs` + fitur `pos-prevnext-wordfrom`, `representation_crf/check_entity/fuzzywuzzy`, Gradio app.

**Struktur folder (identik di Colab & lokal, ganti 1 baris `ROOT`):**
`IE_project/input_pdf/PDF_Putusan, output_txt/TXT_Putusan, output_csv/, models/, code/`


## V0 — Konfigurasi path tunggal (SATU-SATUNYA baris yang diganti per tempat run)

In [1]:
!pip install PyPDF2 tqdm pandas flair sklearn-crfsuite fuzzywuzzy python-Levenshtein gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 26.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
!nvidia-smi

In [4]:
import tensorflow as tf
print("GPU:", tf.test.gpu_device_name())

GPU: /device:GPU:0


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
from pathlib import Path

# >>> GANTI SALAH SATU SESUAI TEMPAT RUN <<<
ROOT = '/home/hujan/Documents/Kuliah/semester-7/Ekstraksi Informasi/Kode Tugas/MACHINE LEARNING IE/IE_project'  # Lokal Linux (default mesin ini)
# ROOT = r'D:/Kuliah/IE_project'              # Lokal Windows
# ROOT = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project'  # Colab + Google Drive (aktifkan bila run di Colab)

DIR_PDF   = str(Path(ROOT) / 'input_pdf' / 'PDF_Putusan')   # 258 PDF sumber
DIR_TXT   = str(Path(ROOT) / 'output_txt' / 'TXT_Putusan')   # 258 TXT hasil
DIR_CSV   = str(Path(ROOT) / 'output_csv')                  # semua CSV out
DIR_MODEL = str(Path(ROOT) / 'models')                      # best-model.pt, *.pkl

for d in [DIR_PDF, DIR_TXT, DIR_CSV, DIR_MODEL]:
    os.makedirs(d, exist_ok=True)
    print(('OK  ' if os.path.isdir(d) else 'FAIL') + f' {d}')
print('\n[INFO] Semua cell di bawah memakai 4 variabel ini. Tidak ada path absolut lain.')


OK   /home/hujan/Documents/Kuliah/semester-7/Ekstraksi Informasi/Kode Tugas/MACHINE LEARNING IE/IE_project/input_pdf/PDF_Putusan
OK   /home/hujan/Documents/Kuliah/semester-7/Ekstraksi Informasi/Kode Tugas/MACHINE LEARNING IE/IE_project/output_txt/TXT_Putusan
OK   /home/hujan/Documents/Kuliah/semester-7/Ekstraksi Informasi/Kode Tugas/MACHINE LEARNING IE/IE_project/output_csv
OK   /home/hujan/Documents/Kuliah/semester-7/Ekstraksi Informasi/Kode Tugas/MACHINE LEARNING IE/IE_project/models

[INFO] Semua cell di bawah memakai 4 variabel ini. Tidak ada path absolut lain.


## V1 — Verifikasi input sebelum proses lama berjalan

In [7]:
import glob
pdfs = sorted(glob.glob(os.path.join(DIR_PDF, '*.pdf')))
print(f'[CHECK] PDF ditemukan: {len(pdfs)} (target 258)')
for f in pdfs[:5]:
    print('  -', os.path.basename(f))
bm = os.path.join(DIR_MODEL, 'best-model.pt')
print(f"[CHECK] best-model.pt: {'ADA' if os.path.isfile(bm) else 'TIDAK ADA -> ' + bm}")
print('[CHECK] File baseline ML2 (opsional, untuk referensi):')
print('  Data/data_mlformat_clean.csv untuk melihat format target.')
assert len(pdfs) > 0, 'DIR_PDF kosong! Cek ROOT.'


[CHECK] PDF ditemukan: 258 (target 258)
  - 1002_Pid.Sus-LH_2026_PN Sby.pdf
  - 1013_Pid.B_2026_PN Sby.pdf
  - 1016_Pid.B_2026_PN Sby.pdf
  - 1021_Pid.Sus_2026_PN Sby.pdf
  - 1050_Pid.Sus_2026_PN Sby.pdf
[CHECK] best-model.pt: TIDAK ADA -> /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/models/best-model.pt
[CHECK] File baseline ML2 (opsional, untuk referensi):
  Data/data_mlformat_clean.csv untuk melihat format target.


## S — Pengganti scraping (cell 0–16 asli DIBUANG)
Datamu sudah ada lokal di `DIR_PDF`, jadi tidak ada `scraping_putusan/get_data/zip_dokumen`. Fungsi baca PDF asli dipertahankan + cleaner tambahan Surabaya.

In [8]:
import re
import PyPDF2
import pandas as pd
from tqdm import tqdm

def clean_page_sby(text):
    """Asli cell 89 FinalCode + tambahan watermark pinggir Surabaya."""
    if not text:
        return ''
    text = text.replace("Mahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nDirektori Putusan Mahkamah Agung Republik Indonesia\nputusan.mahkamahagung.go.id\n", "")
    text = text.replace("\nDisclaimer\nKepaniteraan Mahkamah Agung Republik Indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen Mahkamah Agung untuk pelayanan publik, transparansi dan akuntabilitas\npelaksanaan fungsi peradilan. Namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu.\nDalam hal Anda menemukan inakurasi informasi yang termuat pada situs ini atau informasi yang seharusnya ada, namun belum tersedia, maka harap segera hubungi Kepaniteraan Mahkamah Agung RI melalui :\nEmail : kepaniteraan@mahkamahagung.go.id", "")
    text = text.replace("Telp : 021-384 3348 (ext.318)", "")
    text = re.sub(r'\nHalaman \d+ dari \d+ .*', '', text)
    text = re.sub(r'Halaman \d+ dari \d+ .*', '', text)
    text = re.sub(r'\nHal\. \d+ dari \d+ .*', '', text)
    text = re.sub(r'Hal\. \d+ dari \d+ .*', '', text)
    # Tambahan Surabaya: watermark kolom pinggir hasil pdftotext -layout
    text = re.sub(r'^(hk|am|In|gu|do|ne|si|R|a|ep|ub|lik|ah|ka)\s*$', '', text, flags=re.MULTILINE)
    return text.strip()

def read_pdf_sby(file_pdf):
    """Asli cell 89 FinalCode (tanpa requests-exception yang salah pakai)."""
    pdf_text = ''
    with open(file_pdf, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            pdf_text += ' ' + clean_page_sby(page.extract_text() or '')
    return pdf_text.strip()

def clean_text_sby(text):
    """Asli cell 91 FinalCode (clean_text v1)."""
    text = text.replace('P U T U S A N', 'PUTUSAN').replace('T erdakwa', 'Terdakwa').replace('T empat', 'Tempat').replace('T ahun', 'Tahun')
    text = text.replace('P  E  N  E  T  A  P  A  N', 'PENETAPAN').replace('J u m l a h', 'Jumlah').replace('M E N G A D I L I', 'MENGADILI').replace('M E N E T A P K A N', 'MENETAPKAN')
    text = re.sub(r'Halaman \d+', ' ', text)
    text = text.replace('\uf0d8', '').replace('\uf0b7', '').replace('\uf0fc', ' ').replace('\n', ' ')
    text = re.sub(r'([“”"])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

JENIS_RE = re.compile(r'_(Pid\.B|Pid\.Sus-LH|Pid\.Sus-TPK|Pid\.Pra|Pid\.Sus|Pdt\.Sus-[A-Za-z\-]+|Pdt\.G\.S|Pdt\.G|Pdt\.P|Pdt\.Bth)_')

def parse_namafile(nama):
    m = JENIS_RE.search('_' + nama + '_')
    jenis = m.group(1) if m else 'LAIN'
    th = re.search(r'_(19|20\d{2})_', '_' + nama + '_')
    peng = 'PN Niaga Surabaya' if 'Niaga' in nama else 'PN Surabaya'
    return jenis, (th.group(1) if th else ''), peng

print('[LOAD] S0 OK: read_pdf_sby, clean_text_sby, parse_namafile siap.')


[LOAD] S0 OK: read_pdf_sby, clean_text_sby, parse_namafile siap.


In [9]:
# S1 — Scan DIR_PDF -> data_all_surabaya.csv (pengganti data_all.csv cell 17/26 asli)
rows = []
for fp in tqdm(sorted(__import__('glob').glob(os.path.join(DIR_PDF, '*.pdf'))), desc='Scan PDF'):
    nama = os.path.basename(fp)
    jenis, tahun, peng = parse_namafile(nama)
    rows.append({'nama_file': nama, 'jenis_perkara': jenis, 'tahun': tahun,
                 'pengadilan': peng, 'pdf_path': fp})
df_scan = pd.DataFrame(rows)
out_scan = os.path.join(DIR_CSV, 'data_all_surabaya.csv')
# content_mentah/content_bersih diisi tahap S2 agar S1 tetap cepat
df_scan.to_csv(out_scan, index=False)
print(f'[SAVE] {out_scan} ({len(df_scan)} baris)')
print(df_scan['jenis_perkara'].value_counts().to_string())


Scan PDF: 100%|██████████| 258/258 [00:00<00:00, 148664.71it/s]

[SAVE] /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/output_csv/data_all_surabaya.csv (258 baris)
jenis_perkara
Pdt.P                 97
Pid.Sus               51
Pid.B                 44
Pdt.G                 37
Pdt.Sus-PKPU           7
Pdt.Sus-PHI            6
Pdt.Bth                4
Pid.Sus-LH             3
Pdt.G.S                3
Pdt.Sus-HKI            2
Pid.Sus-TPK            1
Pid.Pra                1
Pdt.Sus-Pailit-GLL     1
LAIN                   1


In [10]:
# S2 — Konversi setiap PDF -> TXT (folder output_txt/TXT_Putusan) + log_konversi.csv
# Bisa di-run di lokal (cepat, tanpa GPU). Resume otomatis: file TXT yang ada dilewati.
df_scan = pd.read_csv(os.path.join(DIR_CSV, 'data_all_surabaya.csv'))
log = []
for _, r in tqdm(df_scan.iterrows(), total=len(df_scan), desc='PDF->TXT'):
    txt_name = os.path.splitext(r['nama_file'])[0] + '.txt'
    txt_path = os.path.join(DIR_TXT, txt_name)
    if os.path.isfile(txt_path) and os.path.getsize(txt_path) > 0:
        log.append({'nama_file': r['nama_file'], 'txt_path': txt_path, 'status': 'skip-ada'})
        continue
    try:
        with open(r['pdf_path'], 'rb') as f:
            n_hal = len(PyPDF2.PdfReader(f).pages)
        mentah = read_pdf_sby(r['pdf_path'])
        bersih = clean_text_sby(mentah)
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(bersih)
        log.append({'nama_file': r['nama_file'], 'txt_path': txt_path, 'halaman': n_hal,
                    'n_char': len(bersih), 'n_kata': len(bersih.split()), 'status': 'ok'})
    except Exception as e:
        log.append({'nama_file': r['nama_file'], 'txt_path': txt_path, 'status': f'gagal: {e}'})
df_log = pd.DataFrame(log)
df_log.to_csv(os.path.join(DIR_CSV, 'log_konversi.csv'), index=False)
print(f"[SAVE] TXT: {sum(df_log['status'] == 'ok')} ok, {sum(df_log['status'] == 'skip-ada')} skip")
print(f"[SAVE] {os.path.join(DIR_CSV, 'log_konversi.csv')}")
df_log.head()


PDF->TXT: 100%|██████████| 258/258 [03:34<00:00,  1.20it/s]

[SAVE] TXT: 258 ok, 0 skip
[SAVE] /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/output_csv/log_konversi.csv


,nama_file,txt_path,halaman,n_char,n_kata,status
0,1002_Pid.Sus-LH_2026_PN Sby.pdf,/content/drive/MyDrive/Kuliah/Semester 7/Ekstr...,34,72030,10223,ok
1,1013_Pid.B_2026_PN Sby.pdf,/content/drive/MyDrive/Kuliah/Semester 7/Ekstr...,13,25477,3605,ok
2,1016_Pid.B_2026_PN Sby.pdf,/content/drive/MyDrive/Kuliah/Semester 7/Ekstr...,16,34417,5405,ok
3,1021_Pid.Sus_2026_PN Sby.pdf,/content/drive/MyDrive/Kuliah/Semester 7/Ekstr...,16,34086,5025,ok
4,1050_Pid.Sus_2026_PN Sby.pdf,/content/drive/MyDrive/Kuliah/Semester 7/Ekstr...,83,141819,21875,ok


## L — Bootstrap label regex unified (pengganti Excel `entities_all.xlsx` + `generate_label` cell 43 asli)
Excel 12 kolom pidana tidak ada untuk Surabaya, jadi diganti 15 pola regex. `labeling_token/anotasi/handleDuplicatePena/countLabel` dipertahankan dari cell 45–48 asli.

In [11]:
# L0 — Kamus regex unified: 6 lintas-jenis + 9 spesifik jenis. Nilai dipakai sebagai surface form BIO.
POLA = {
    'VERN': r'\d+\/(?:Pid\.B|Pid\.Sus(?:-LH|-TPK)?|Pid\.Pra|Pdt\.G(?:\.S)?|Pdt\.P|Pdt\.Bth|Pdt\.Sus-[A-Za-z\-]+)\/\d{4}\/PN[^;\n]{0,20}',
    'TIMV': r'\d{1,2}\s+(?:Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|Nopember|November|Desember)\s+\d{4}',
    'JUDP': r'Hakim Ketua\s*:\s*([A-Z][A-Za-z\.\s,]+?)(?=\s*(?:Hakim|Panitera|\d+\.|$))',
    'JUDG': r'Hakim Anggota[^:]*:\s*([A-Z][A-Za-z\.\s,]+?)(?=\s*(?:Hakim|Panitera|\d+\.|$))',
    'REGI': r'Panitera(?: Pengganti)?\s*:\s*([A-Z][A-Za-z\.\s,]+?)(?=\s*(?:\d+\.|$))',
    'ARTV': r'Pasal\s+\d+[^\n;]{0,80}(?:UU\s*(?:No\.?|Nomor)?\s*\d+[^\n;]{0,40}|KUHP|KUHAP)',
    'DEFN': r'Terdakwa\s+([A-Z][A-Z\s\.\']+(?:Bin|Binti|Als)\s+[A-Z\s\.\']+)',
    'PROS': r'(?:Jaksa Penuntut Umum|Penuntut Umum)\s*:\s*([A-Z][A-Za-z\s\.,]+)',
    'CRIA': r'Dakwaan\s+(?:Tunggal|Alternatif|Subsidiaritas)[^;]{0,200}',
    'PENA': r'[Tt]untutan[^;]{0,120}pidana[^;]{0,120}',
    'PUNI': r'Menjatuhkan pidana[^;]{0,150}',
    'BB':   r'[Bb]arang bukti[^:]*:\s*([^;]{5,200})',
    'PENGG': r'antara\s*:\s*(.+?)\s+sebagai\s+PENGGUGAT',
    'TERG': r'[Ll]awan\s*:\s*(.+?)\s+sebagai\s+TERGUGAT',
    'PEMOHON': r'Pemohon\s*:\s*([A-Z][A-Z\s]+)',
    'KUASA': r'[Kk]uasa [Hh]ukum[^:]*:\s*([^;]{5,200})',
    'NIK':  r'\b\d{16}\b',
    'AMAR': r'MENGADILI\s*:(.{20,2000})',
}
LABEL_PER_JENIS = {
    'PID': ['VERN','TIMV','JUDP','JUDG','REGI','ARTV','DEFN','PROS','CRIA','PENA','PUNI','BB'],
    'PDTG': ['VERN','TIMV','JUDP','JUDG','REGI','ARTV','PENGG','TERG','KUASA','AMAR'],
    'PDTP': ['VERN','TIMV','JUDP','JUDG','REGI','PEMOHON','NIK','AMAR'],
    'NIAGA': ['VERN','TIMV','JUDP','JUDG','REGI','ARTV','PENGG','TERG','KUASA','AMAR'],
}
def kelompok(jenis):
    if jenis.startswith('Pid'): return 'PID'
    if jenis in ('Pdt.G','Pdt.Bth','Pdt.G.S'): return 'PDTG'
    if jenis == 'Pdt.P': return 'PDTP'
    return 'NIAGA'

def regex_label(content, jenis):
    lab = {}
    for kode in LABEL_PER_JENIS[kelompok(jenis)]:
        for m in re.finditer(POLA[kode], content):
            surf = (m.group(1) if m.lastindex else m.group(0)).strip().lower()
            surf = re.sub(r'\s+', ' ', re.sub(r'\b(s\.h\.?|m\.h\.?|sh|mh)\b', '', surf)).strip(' ,.')
            if len(surf) >= 3:
                lab[surf] = 'B_' + kode
    return lab

print('[LOAD] L0 OK: 18 pola regex + peta label per kelompok jenis.')


[LOAD] L0 OK: 18 pola regex + peta label per kelompok jenis.


In [12]:
# L1 — labeling_token / anotasi / handleDuplicatePena / countLabel (verbatim cell 45-48 asli).
# Bedanya: generate_label Excel diganti regex_label; dropna() dibuang (sel kosong -> O).
def labeling_token(text, label_dict):
    # Longest-match langsung (pengganti trie): kebal tabrakan kata 'label' dkk.
    tokens = text.lower().split()
    labeled_tokens = ['O'] * len(tokens)
    clean = [re.sub(r'[.,;"]', '', t) for t in tokens]
    phrases = sorted(((k, v) for k, v in label_dict.items() if k),
                     key=lambda kv: len(kv[0].split()), reverse=True)
    for key, value in phrases:
        words = [re.sub(r'[.,;"]', '', w) for w in key.split()]
        if not words or not words[0]:
            continue
        n = len(words)
        for i in range(len(tokens) - n + 1):
            if all(l == 'O' for l in labeled_tokens[i:i+n]) and clean[i:i+n] == words:
                labeled_tokens[i] = value
                for j in range(1, n):
                    labeled_tokens[i+j] = value.replace('B_', 'I_', 1)
    return labeled_tokens

def handleDuplicatePena(sequences, aktif=True):
    if not aktif:
        return sequences
    countPena = 0
    for sequence in sequences:
        for i, label in enumerate(sequence):
            if label == 'B_PENA':
                countPena += 1
                if countPena > 1:
                    sequence[i] = 'B_PUNI'
            elif label == 'I_PENA':
                if countPena > 1:
                    sequence[i] = 'I_PUNI'
    return sequences

LABEL_LIST = ['B_ARTV','B_DEFN','B_PROS','B_CRIA','B_JUDP','B_VERN','B_PUNI','B_JUDG','B_REGI','B_TIMV','B_PENA','B_BB','B_PENGG','B_TERG','B_PEMOHON','B_KUASA','B_NIK','B_AMAR']

def anotasi_sby(df_in):
    content_results = pd.DataFrame(columns=['doc','text','label','nama_file','jenis_perkara'])
    labels = []
    for doc_idx, (_, row) in tqdm(enumerate(df_in.iterrows()), total=len(df_in), desc='Annotating'):
        lab = regex_label(row['content_bersih'], row['jenis_perkara'])
        labels.append(lab)
        toks = labeling_token(row['content_bersih'], lab)
        tmp = pd.DataFrame({'doc': [f'doc:{doc_idx+1}'], 'text': [row['content_bersih'].split()],
                            'label': [toks], 'nama_file': [row['nama_file']],
                            'jenis_perkara': [row['jenis_perkara']]})
        tmp['label'] = handleDuplicatePena(tmp['label'], aktif=row['jenis_perkara'].startswith('Pid'))
        content_results = pd.concat([content_results, tmp], ignore_index=True)
    return content_results, labels

print('[LOAD] L1 OK.')


[LOAD] L1 OK.


In [13]:
# L2 — Jalankan anotasi Surabaya (butuh TXT selesai). Output: new_df + coverage.
df_txt = pd.read_csv(os.path.join(DIR_CSV, 'log_konversi.csv'))
df_scan = pd.read_csv(os.path.join(DIR_CSV, 'data_all_surabaya.csv'))
rows = []
for _, r in df_scan.iterrows():
    tp = os.path.join(DIR_TXT, os.path.splitext(r['nama_file'])[0] + '.txt')
    with open(tp, encoding='utf-8') as f:
        rows.append({'nama_file': r['nama_file'], 'jenis_perkara': r['jenis_perkara'],
                     'content_bersih': f.read()})
df_in = pd.DataFrame(rows)
new_df, labels = anotasi_sby(df_in)
n_tok = sum(len(t) for t in new_df['text'])
n_lab = sum(1 for labs in new_df['label'] for l in labs if l != 'O')
print(f'[INFO] Dok: {len(new_df)}, token: {n_tok}, token berlabel: {n_lab} ({n_lab/max(n_tok,1)*100:.2f}%)')
print('[CONTOH]', list(labels[0].items())[:5])


Annotating: 100%|██████████| 258/258 [01:52<00:00,  2.30it/s]

[INFO] Dok: 258, token: 1508418, token berlabel: 63716 (4.22%)
[CONTOH] [('1002/pid.sus-lh/2026/pn sby demi keadilan b', 'B_VERN'), ('1002/pdt.sus-lh/2026/pn sby tertanggal 15 j', 'B_VERN'), ('1002/pid.sus-lh/2026/pn sby tanggal 23 juni', 'B_VERN'), ('29 september 1993', 'B_TIMV'), ('17 april 2026', 'B_TIMV')]


## F — Tokenisasi + fragmentasi (verbatim cell 52 & 56 asli, + kolom asal)

In [14]:
# F0 — split_token (cell 52 asli) lalu fragmentasi per ';' (cell 56 asli).
def split_token(dataset):
    dataset_copy = dataset.copy()
    for i, row in tqdm(dataset_copy.iterrows(), total=len(dataset_copy), desc='Processing rows'):
        for index, (text, tag) in enumerate(zip(row['text'], row['label'])):
            token = re.findall(r'[^\/\,\.():;]+|[\/\.,():;]', text)
            if tag != 'O':
                add_label = []
                for k in range(len(token)):
                    if 'B' in tag and not add_label:
                        add_label.append(tag)
                    else:
                        add_label.append(tag.replace('B', 'I') if token[k] != ';' else 'O')
                del row['text'][index]; del row['label'][index]
                row['text'][index:index] = token; row['label'][index:index] = add_label
            else:
                add_label = ['O'] * len(token)
                del row['text'][index]; del row['label'][index]
                row['text'][index:index] = token; row['label'][index:index] = add_label
    return dataset_copy

splited_df = split_token(new_df)
new_rows, fragment_id = [], 1
for _, row in splited_df.iterrows():
    fragments, labs, frag, lab = [], [], [], []
    for word, label_word in zip(row['text'], row['label']):
        if word != ';':
            frag.append(word); lab.append(label_word)
        else:
            if frag:
                fragments.append(frag); labs.append(lab)
            frag, lab = [], []
    if frag:
        fragments.append(frag); labs.append(lab)
    for frag, lab in zip(fragments, labs):
        new_rows.append({'text': frag, 'label': lab, 'doc': row['doc'],
                         'fragment': f'fragment:{fragment_id}', 'nama_file': row['nama_file'],
                         'jenis_perkara': row['jenis_perkara']})
        fragment_id += 1
df_perfrag = pd.DataFrame(new_rows)
df_pertoken = pd.DataFrame(
    [{'text': token, 'label': label, 'doc': row['doc'], 'fragment': row['fragment'],
      'nama_file': row['nama_file'], 'jenis_perkara': row['jenis_perkara']}
     for _, row in df_perfrag.iterrows() for token, label in zip(row['text'], row['label'])])
df_pertoken = df_pertoken.rename(columns={'text': 'token'})
out_nopos = os.path.join(DIR_CSV, 'ml_format_nopos_surabaya.csv')
df_pertoken.to_csv(out_nopos, index=False)
print(f'[SAVE] {out_nopos} ({len(df_pertoken)} baris)')
print(df_pertoken['label'].value_counts().head(10).to_string())


Processing rows: 100%|██████████| 258/258 [00:52<00:00,  4.92it/s]


[SAVE] /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/output_csv/ml_format_nopos_surabaya.csv (1888047 baris)
label
O          1803166
I_TIMV       24638
I_BB         16302
I_VERN       14857
B_TIMV       11164
I_II          4699
I_ARTV        3405
I_DEFN        2963
B_VERN        1382
I_KUASA       1042


## P — POS tagging (TERMASUK training GPU, verbatim cell 60 & 63 asli)
Urutan: **P1-training (GPU)** → P2-inferensi batch + checkpoint → P3-parse ke CSV.
Training `UD_INDONESIAN + StackedEmbeddings id-crawl/id 1.1GB, BiLSTM-256 + CRF, 10 epoch` WAJIB di-run di Colab GPU (Runtime → T4) agar setiap cell punya output untuk bukti PDF. Bila runtime putus (catatan cell 64 asli), ulangi P1 — checkpoint P2 membuat inferensi tetap bisa dilanjutkan.

### P1 — Training POS tagger (GPU, verbatim FinalCode cell 60 & 63 + POS tagging.ipynb)
Jalankan di **Colab GPU**. Output log per epoch (loss + DEV micro-F1) diperlukan untuk PDF bukti. Estimasi 1–3 jam di T4 gratis.

In [3]:
!pip install "flair[word-embeddings]" gensim

In [4]:
# P1a — import (verbatim FinalCode cell 60 asli)
# pip (jalankan sekali di Colab): pip install flair tqdm
import flair.datasets
from flair.embeddings import TokenEmbeddings, WordEmbeddings, StackedEmbeddings
from typing import List
from flair.data import Sentence
from flair.models import SequenceTagger
from flair.trainers import ModelTrainer
import shutil
print('[LOAD] P1a OK: flair imports siap.')


[LOAD] P1a OK: flair imports siap.


In [7]:
# P1b — training (verbatim FinalCode cell 63 asli; hanya tujuan salinan diganti Drive -> DIR_MODEL agar tidak ambigu)
#load data bahasa indonesia
corpus = flair.datasets.UD_INDONESIAN()

tag_type = 'upos'
tag_dictionary = corpus.make_tag_dictionary(tag_type=tag_type)

#mengubah ke embedding
embedding_types: List[TokenEmbeddings] = [
 WordEmbeddings('id-crawl'),
 WordEmbeddings('id'),
]
embeddings: StackedEmbeddings = StackedEmbeddings(embeddings=embedding_types)

tagger: SequenceTagger = SequenceTagger(hidden_size=256,
                                        embeddings=embeddings,
                                        tag_dictionary=tag_dictionary,
                                        tag_type=tag_type,
                                        use_crf=True)

#training model pos
trainer: ModelTrainer = ModelTrainer(tagger, corpus)
trainer.train('resources/taggers/example-universal-pos',
              learning_rate=0.1,
              mini_batch_size=32,
              max_epochs=10)

# Menyalin model ke DIR_MODEL (asli: Google Drive) — path tidak ambigu
model_path = 'resources/taggers/example-universal-pos/best-model.pt'
drive_model_path = f"{DIR_MODEL}/best-model.pt"
shutil.copy(model_path, drive_model_path)
print(f'[SAVE] {drive_model_path}')


2026-10-09 22:50:25,883 Reading data from /root/.flair/datasets/ud_indonesian
2026-10-09 22:50:25,884 Train: /root/.flair/datasets/ud_indonesian/id_gsd-ud-train.conllu
2026-10-09 22:50:25,886 Dev: /root/.flair/datasets/ud_indonesian/id_gsd-ud-dev.conllu
2026-10-09 22:50:25,887 Test: /root/.flair/datasets/ud_indonesian/id_gsd-ud-test.conllu


/tmp/ipykernel_8596/896403830.py:6: DeprecationWarning: Call to deprecated method make_tag_dictionary. (Use 'make_label_dictionary' instead.) -- Deprecated since version 0.8.
  tag_dictionary = corpus.make_tag_dictionary(tag_type=tag_type)


2026-10-09 22:50:52,583 SequenceTagger predicts: Dictionary with 20 tags: O, PROPN, AUX, DET, NOUN, PRON, VERB, ADP, PUNCT, ADV, CCONJ, SCONJ, NUM, ADJ, PART, SYM, INTJ, X, <START>, <STOP>
2026-10-09 22:50:52,681 ----------------------------------------------------------------------------------------------------
2026-10-09 22:50:52,682 Model: "SequenceTagger(
  (embeddings): StackedEmbeddings(
    (list_embedding_0): WordEmbeddings(
      'id-crawl'
      (embedding): Embedding(1000000, 300)
    )
    (list_embedding_1): WordEmbeddings(
      'id'
      (embedding): Embedding(300686, 300)
    )
  )
  (word_dropout): WordDropout(p=0.05)
  (locked_dropout): LockedDropout(p=0.5)
  (embedding2nn): Linear(in_features=600, out_features=600, bias=True)
  (rnn): LSTM(600, 256, batch_first=True, bidirectional=True)
  (linear): Linear(in_features=512, out_features=20, bias=True)
  (loss_function): ViterbiLoss()
  (crf): CRF()
)"
2026-10-09 22:50:52,684 -------------------------------------------

/usr/local/lib/python3.13/dist-packages/flair/trainers/trainer.py:107: UserWarning: There should be no best model saved at epoch 1 except there is a model from previous trainings in your training folder. All previous best models will be deleted.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/flair/trainers/trainer.py:545: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and flair.device.type != "cpu")


2026-10-09 22:50:54,663 epoch 1 - iter 14/141 - loss 2.63348032 - time (sec): 1.96 - samples/sec: 5313.72 - lr: 0.100000 - momentum: 0.000000
2026-10-09 22:50:56,532 epoch 1 - iter 28/141 - loss 2.39010371 - time (sec): 3.82 - samples/sec: 5275.81 - lr: 0.100000 - momentum: 0.000000
2026-10-09 22:50:59,011 epoch 1 - iter 42/141 - loss 2.16252565 - time (sec): 6.30 - samples/sec: 4688.91 - lr: 0.100000 - momentum: 0.000000
2026-10-09 22:51:01,148 epoch 1 - iter 56/141 - loss 1.98635111 - time (sec): 8.44 - samples/sec: 4654.76 - lr: 0.100000 - momentum: 0.000000
2026-10-09 22:51:02,810 epoch 1 - iter 70/141 - loss 1.84182522 - time (sec): 10.10 - samples/sec: 4842.31 - lr: 0.100000 - momentum: 0.000000
2026-10-09 22:51:04,369 epoch 1 - iter 84/141 - loss 1.72489320 - time (sec): 11.66 - samples/sec: 5011.10 - lr: 0.100000 - momentum: 0.000000
2026-10-09 22:51:06,113 epoch 1 - iter 98/141 - loss 1.62862804 - time (sec): 13.41 - samples/sec: 5090.99 - lr: 0.100000 - momentum: 0.000000
202

...

2026-10-09 23:03:36,112 DEV : loss 0.2326812744140625 - f1-score (micro avg)  0.9228
2026-10-09 23:03:36,170  - 0 epochs without improvement
2026-10-09 23:03:36,172 saving best model


2026-10-09 23:06:06,256 ----------------------------------------------------------------------------------------------------
2026-10-09 23:06:06,265 Loading model from best epoch ...
2026-10-09 23:06:17,197 SequenceTagger predicts: Dictionary with 20 tags: O, PROPN, AUX, DET, NOUN, PRON, VERB, ADP, PUNCT, ADV, CCONJ, SCONJ, NUM, ADJ, PART, SYM, INTJ, X, <START>, <STOP>


100%|██████████| 9/9 [00:02<00:00,  4.12it/s]

2026-10-09 23:06:20,499 
Results:
- F-score (micro) 0.9268
- F-score (macro) 0.873
- Accuracy 0.9268

By class:
              precision    recall  f1-score   support

        NOUN     0.8997    0.8940    0.8968      2509
       PROPN     0.9012    0.9450    0.9226      2162
       PUNCT     0.9951    0.9988    0.9969      1614
        VERB     0.9450    0.9299    0.9374      1256
         ADP     0.9406    0.9515    0.9460      1114
        PRON     0.9515    0.9752    0.9632       644
         ADJ     0.8386    0.7680    0.8017       487
         NUM     0.9866    0.9532    0.9696       385
       CCONJ     0.9861    0.9725    0.9793       364
         ADV     0.7994    0.8182    0.8087       341
         DET     0.9676    0.8846    0.9243       338
         AUX     0.9571    0.9738    0.9654       229
       SCONJ     0.8539    0.7835    0.8172       194
        PART     0.8958    0.9663    0.9297        89
           X     0.5000    0.1250    0.2000        24
         SYM     1.0000

[SAVE] /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/models/best-model.pt


In [8]:
# P2 — Inferensi POS per token, batch + checkpoint agar aman di Colab gratis.
# pip (jalankan sekali di Colab): pip install flair tqdm
from flair.data import Sentence
from flair.models import SequenceTagger
import pandas as pd
from tqdm import tqdm

df = pd.read_csv(os.path.join(DIR_CSV, 'ml_format_nopos_surabaya.csv'))
tokens = df['token'].tolist()
tagger = SequenceTagger.load(os.path.join(DIR_MODEL, 'best-model.pt'))
BATCH, ckpt_dir = 50000, os.path.join(DIR_CSV, 'pos_ckpt')
os.makedirs(ckpt_dir, exist_ok=True)
f_pos_all = []
for b in range(0, len(tokens), BATCH):
    ck = os.path.join(ckpt_dir, f'pos_{b}_{min(b+BATCH, len(tokens))}.txt')
    if os.path.isfile(ck):
        with open(ck, encoding='utf-8') as f:
            f_pos_all.extend([l.rstrip('\n') for l in f])
        print(f'[SKIP] checkpoint {os.path.basename(ck)}')
        continue
    buf = []
    for tok in tqdm(tokens[b:b+BATCH], desc=f'POS {b}-{min(b+BATCH, len(tokens))}'):
        s = Sentence(str(tok))
        tagger.predict(s)
        buf.append(s.to_tagged_string())
    with open(ck, 'w', encoding='utf-8') as f:
        f.write('\n'.join(buf))
    f_pos_all.extend(buf)
    print(f'[SAVE] {ck}')
print(f'[INFO] f_pos terkumpul: {len(f_pos_all)}')


2026-10-09 23:09:26,198 SequenceTagger predicts: Dictionary with 20 tags: O, PROPN, AUX, DET, NOUN, PRON, VERB, ADP, PUNCT, ADV, CCONJ, SCONJ, NUM, ADJ, PART, SYM, INTJ, X, <START>, <STOP>


POS 0-50000: 100%|██████████| 50000/50000 [02:06<00:00, 394.92it/s]


[SAVE] /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/output_csv/pos_ckpt/pos_0_50000.txt


POS 50000-100000: 100%|██████████| 50000/50000 [02:06<00:00, 395.46it/s]


[SAVE] /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/output_csv/pos_ckpt/pos_50000_100000.txt


...

[SAVE] /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/output_csv/pos_ckpt/pos_1750000_1800000.txt


POS 1800000-1850000: 100%|██████████| 50000/50000 [02:04<00:00, 400.19it/s]


[SAVE] /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/output_csv/pos_ckpt/pos_1800000_1850000.txt


POS 1850000-1888047: 100%|██████████| 38047/38047 [01:36<00:00, 393.93it/s]

[SAVE] /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/output_csv/pos_ckpt/pos_1850000_1888047.txt
[INFO] f_pos terkumpul: 1888047


In [10]:
# P3 — Parse tagged_string -> kolom pos -> data_mlformat_surabaya.csv (format final ML2).
import re

pos = []
for s in tqdm(f_pos_all, desc='Parse POS'):
    m = re.split(r'/', s)
    pos.append(m[-1][:-1] if len(m) > 1 else 'X')
assert len(pos) == len(df), 'Jumlah POS != jumlah token!'
df['pos'] = pos
out_ml = os.path.join(DIR_CSV, 'data_mlformat_surabaya.csv')
df.to_csv(out_ml, index=False)
print(f'[SAVE] {out_ml} ({len(df)} baris x {list(df.columns)})')
print(df['pos'].value_counts().head(10).to_string())


Parse POS: 100%|██████████| 1888047/1888047 [00:07<00:00, 248727.58it/s]


[SAVE] /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/output_csv/data_mlformat_surabaya.csv (1888047 baris x ['token', 'label', 'doc', 'fragment', 'nama_file', 'jenis_perkara', 'pos'])
pos
NOUN     418817
PUNCT    385089
PROPN    329310
NUM      197581
VERB     150115
ADP      146269
SCONJ     53240
PRON      40329
CCONJ     39855
ADJ       33061


## M — Feature engineering + modelling CRF (disalin dari `Testing/600crf-pos-prevnext-wordfrom.ipynb`)
Dijalankan di Kaggle/Colab RAM besar seperti catatan cell 68–72 asli. Bukan GPU.

In [11]:
# M0 — FragmentGetter + token2features LENGKAP (token+pos+prev1/2+next1/2+wordshape+BOF/EOF).
# pip (sekali): pip install sklearn-crfsuite
df = pd.read_csv(os.path.join(DIR_CSV, 'data_mlformat_surabaya.csv'))
df = df.fillna('')

class FragmentGetter(object):
    def __init__(self, data):
        self.n_frag = 1
        self.data = data
        self.empty = False
        agg_func = lambda f: [(t, p, l) for t, p, l in zip(f['token'].values.tolist(),
                                                            f['pos'].values.tolist(),
                                                            f['label'].values.tolist())]
        self.grouped = self.data.groupby('fragment').apply(agg_func)
        self.fragments = [f for f in self.grouped]

def token2features(frag, i):
    token, postag = frag[i][0], frag[i][1]
    features = {'bias': 1.0, 'token': token, 'token.lower()': token.lower(),
                'token.isupper()': token.isupper(), 'token.istitle()': token.istitle(),
                'token.isdigit()': token.isdigit(), 'postag': postag}
    if i > 0:
        features.update({'prev1': frag[i-1][0], 'prev1.lower()': frag[i-1][0].lower(),
                         'prev1.isupper()': frag[i-1][0].isupper(), 'prev1.istitle()': frag[i-1][0].istitle(),
                         'prev1.isdigit()': frag[i-1][0].isdigit(), 'prev1.postag': frag[i-1][1]})
    else:
        features['BOF'] = True
    if i > 1:
        features.update({'prev2': frag[i-2][0], 'prev2.lower()': frag[i-2][0].lower(),
                         'prev2.isupper()': frag[i-2][0].isupper(), 'prev2.istitle()': frag[i-2][0].istitle(),
                         'prev2.isdigit()': frag[i-2][0].isdigit(), 'prev2.postag': frag[i-2][1]})
    if i < len(frag) - 1:
        features.update({'next1': frag[i+1][0], 'next1.lower()': frag[i+1][0].lower(),
                         'next1.isupper()': frag[i+1][0].isupper(), 'next1.istitle()': frag[i+1][0].istitle(),
                         'next1.isdigit()': frag[i+1][0].isdigit(), 'next1.postag': frag[i+1][1]})
    else:
        features['EOF'] = True
    if i < len(frag) - 2:
        features.update({'next2': frag[i+2][0], 'next2.lower()': frag[i+2][0].lower(),
                         'next2.isupper()': frag[i+2][0].isupper(), 'next2.istitle()': frag[i+2][0].istitle(),
                         'next2.isdigit()': frag[i+2][0].isdigit(), 'next2.postag': frag[i+2][1]})
    return features

def frag2features(frag):
    return [token2features(frag, i) for i in range(len(frag))]
def frag2labels(frag):
    return [label for token, postag, label in frag]

X_crf = df.drop(['doc'], axis=1)
getter = FragmentGetter(X_crf)
fragments = getter.fragments
X = [frag2features(f) for f in fragments]
y = [frag2labels(f) for f in fragments]
print(f'[INFO] fragmen: {len(fragments)}, contoh label: {y[0][:8]}')


/tmp/ipykernel_8596/3628641741.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  self.grouped = self.data.groupby('fragment').apply(agg_func)


[INFO] fragmen: 27778, contoh label: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [12]:
# M1 — Split level dokumen 70/30 STRATIFIED jenis_perkara (perbaikan dari cell 80 asli yang acak global)
# agar Pdt.P yang pendek tidak menenggelamkan Pdt.G/Niaga yang panjang.
import time
import numpy as np
import sklearn_crfsuite

frag_doc = df.drop_duplicates('fragment')[['fragment','doc','jenis_perkara']]
docs = df['doc'].unique().tolist()
rng = np.random.RandomState(42)
train_docs = []
for jk, g in df.drop_duplicates('doc')[['doc','jenis_perkara']].groupby('jenis_perkara'):
    d = g['doc'].tolist(); rng.shuffle(d)
    train_docs += d[:int(len(d)*0.7)]
train_frags = set(frag_doc[frag_doc['doc'].isin(train_docs)]['fragment'])
idx = [i for i, f in enumerate(frag_doc['fragment']) if f in train_frags]
X_train = [X[i] for i in idx]; y_train = [y[i] for i in idx]
X_test = [X[i] for i in range(len(X)) if i not in set(idx)]
y_test = [y[i] for i in range(len(y)) if i not in set(idx)]
print(f'[INFO] dok train {len(train_docs)}/{len(docs)}, frag train {len(X_train)}/{len(X)}')

crf = sklearn_crfsuite.CRF(algorithm='lbfgs', c1=0.1, c2=0.1, max_iterations=100,
                           all_possible_transitions=True)
t0 = time.time(); crf.fit(X_train, y_train)
print(f'[INFO] train {time.time()-t0:.1f} detik (acuan Testing: 349-725 dtk).')
import joblib
mp = os.path.join(DIR_MODEL, '600crf_surabaya.pkl')
joblib.dump(crf, mp)
print(f'[SAVE] {mp}')


[INFO] dok train 172/258, frag train 17829/27778
[INFO] train 1322.1 detik (acuan Testing: 349-725 dtk).
[SAVE] /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/models/600crf_surabaya.pkl


In [13]:
# M2 — Evaluasi global + per kelompok jenis (pengganti cell 81 asli yang 1 angka saja).
from sklearn.metrics import classification_report
flat_t = [l for s in y_test for l in s]; flat_p = [l for s in crf.predict(X_test) for l in s]
print(classification_report(flat_t, flat_p, digits=4))
with open(os.path.join(DIR_CSV, 'eval_report.txt'), 'w') as f:
    f.write(classification_report(flat_t, flat_p, digits=4))
print(f"[SAVE] {os.path.join(DIR_CSV, 'eval_report.txt')}")
print('[EKSPEKTASI] Pidana~0.9 / Pdt.P~0.8 / Pdt.G+Niaga~0.6-0.7 (lihat analisa kecocokan).')


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

      B_ARTV     0.7765    0.5500    0.6439       120
        B_BB     0.7241    0.2763    0.4000       304
      B_CRIA     0.8333    0.5000    0.6250        10
      B_DEFN     0.9439    0.5075    0.6601       199
      B_JUDG     1.0000    0.1429    0.2500         7
     B_KUASA     0.0000    0.0000    0.0000        10
       B_NIK     0.8859    0.6111    0.7233       216
   B_PEMOHON     1.0000    0.8077    0.8936        26
      B_PENA     0.0000    0.0000    0.0000         7
      B_PUNI     0.0000    0.0000    0.0000         7
      B_TERG     0.0000    0.0000    0.0000         3
      B_TIMV     0.9801    0.9839    0.9820      4157
      B_VERN     0.8204    0.6036    0.6955       507
      I_ARTV     0.9025    0.5476    0.6816      1386
        I_BB     0.7430    0.2224    0.3423      6111
      I_CRIA     1.0000    0.2625    0.4158        80
      I_DEFN     0.9444    0.5171    0.6683       789
        I_II     0.7429    

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[SAVE] /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/IE_project/output_csv/eval_report.txt
[EKSPEKTASI] Pidana~0.9 / Pdt.P~0.8 / Pdt.G+Niaga~0.6-0.7 (lihat analisa kecocokan).


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## T — Testing PDF baru + agregasi ke CSV entitas-per-kolom (cell 83–99 asli, path diperjelas)

In [2]:
# T0 — representation_crf + check_entity + mapping Indonesia + dedup fuzzy (cell 91-99 asli).
import joblib
from fuzzywuzzy import fuzz

def representation_crf(text, id_fragment, doc_id='doc:1'):
    out = {'doc': [], 'fragment': [], 'token': []}
    for fragment in text.split(';'):
        fragment = re.sub(r'([\/\,\.():;])', r' \1 ', fragment)
        for token in re.findall(r'\S+', fragment):
            out['doc'].append(doc_id); out['fragment'].append(f'fragment:{id_fragment}')
            out['token'].append(token)
    return pd.DataFrame(out)

KEY_MAPPING = {'VERN':'Nomor Putusan','DEFN':'Terdakwa','CRIA':'Tindak Pidana','PENA':'Tuntutan Hukuman',
 'ARTV':'Pasal yang Dilanggar','PUNI':'Putusan Hukuman','JUDP':'Hakim Ketua','JUDG':'Hakim Anggota',
 'TIMV':'Tanggal','REGI':'Panitera','PROS':'Penuntut Umum','BB':'Barang Bukti','PENGG':'Penggugat',
 'TERG':'Tergugat','PEMOHON':'Pemohon','KUASA':'Kuasa Hukum','NIK':'NIK','AMAR':'Amar Putusan'}

def bio_to_entities(tokens, labels):
    ent, cur, cl = {}, '', ''
    for tok, lab in zip(tokens, labels):
        if lab == 'O':
            continue
        tipe = lab.split('_', 1)[1]
        if lab.startswith('B_'):
            if cl:
                ent.setdefault(cl, []).append(cur.strip())
            cur, cl = tok, tipe
        elif lab.startswith('I_') and cl == tipe:
            cur += ' ' + tok
    if cl:
        ent.setdefault(cl, []).append(cur.strip())
    # dedup fuzzy partial_ratio>80, simpan yang terpanjang (cell 97-99 asli)
    for k, vs in list(ent.items()):
        unik = []
        for v in sorted(set(vs), key=len, reverse=True):
            if not any(fuzz.partial_ratio(v, u) > 80 for u in unik):
                unik.append(v)
        ent[k] = unik
    return ent

print('[LOAD] T0 OK: representation_crf, bio_to_entities, KEY_MAPPING (19 entitas).')


[LOAD] T0 OK: representation_crf, bio_to_entities, KEY_MAPPING (19 entitas).


/home/hujan/miniconda3/envs/ekstraksi-informasi/lib/python3.10/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


## TD — Diagnostik 1 dokumen (wajib hijau sebelum T1 penuh)

In [5]:
# TD — Diagnostik 1 dokumen (WAJIB hijau sebelum T1 penuh).
# Membuktikan prediksi tak lagi all-O. Pilih 1 PDF pidana pendek.
import joblib
from collections import Counter
import pandas as pd
import os
import re

CRF_DIAG = joblib.load(os.path.join(DIR_MODEL, '600crf_surabaya.pkl'))
DFML = pd.read_csv(os.path.join(DIR_CSV, 'data_mlformat_surabaya.csv')).fillna('')
POSMAP = {n: list(zip(g['token'].tolist(), g['pos'].tolist()))
          for n, g in DFML.groupby('nama_file')}

def token2features(frag, i):
    # SALINAN PERSIS M0 — jangan diubah agar skema fitur inferensi == training.
    token, postag = frag[i][0], frag[i][1]
    features = {'bias': 1.0, 'token': token, 'token.lower()': token.lower(),
                'token.isupper()': token.isupper(), 'token.istitle()': token.istitle(),
                'token.isdigit()': token.isdigit(), 'postag': postag}
    if i > 0:
        features.update({'prev1': frag[i-1][0], 'prev1.lower()': frag[i-1][0].lower(),
                         'prev1.isupper()': frag[i-1][0].isupper(), 'prev1.istitle()': frag[i-1][0].istitle(),
                         'prev1.isdigit()': frag[i-1][0].isdigit(), 'prev1.postag': frag[i-1][1]})
    else:
        features['BOF'] = True
    if i > 1:
        features.update({'prev2': frag[i-2][0], 'prev2.lower()': frag[i-2][0].lower(),
                         'prev2.isupper()': frag[i-2][0].isupper(), 'prev2.istitle()': frag[i-2][0].istitle(),
                         'prev2.isdigit()': frag[i-2][0].isdigit(), 'prev2.postag': frag[i-2][1]})
    if i < len(frag) - 1:
        features.update({'next1': frag[i+1][0], 'next1.lower()': frag[i+1][0].lower(),
                         'next1.isupper()': frag[i+1][0].isupper(), 'next1.istitle()': frag[i+1][0].istitle(),
                         'next1.isdigit()': frag[i+1][0].isdigit(), 'next1.postag': frag[i+1][1]})
    else:
        features['EOF'] = True
    if i < len(frag) - 2:
        features.update({'next2': frag[i+2][0], 'next2.lower()': frag[i+2][0].lower(),
                         'next2.isupper()': frag[i+2][0].isupper(), 'next2.istitle()': frag[i+2][0].istitle(),
                         'next2.isdigit()': frag[i+2][0].isdigit(), 'next2.postag': frag[i+2][1]})
    return features

def frags_dari_teks(content):
    # Tokenisasi IDENTIK representation_crf, tapi dikelompokkan per fragmen ;.
    frags = []
    for fragment in content.split(';'):
        fragment = re.sub(r'([\/\,\.():;])', r' \1 ', fragment)
        toks = re.findall(r'\S+', fragment)
        if toks:
            frags.append(toks)
    return frags

def pos_untuk_dok(nama_file, frags):
    ref = POSMAP.get(nama_file, [])
    flat_txt = [t for f in frags for t in f]
    if len(ref) == len(flat_txt) and all(a == b for (a, _), b in zip(ref, flat_txt)):
        return [p for _, p in ref], 'reuse-csv'
    # fallback langka: Flair per dokumen (batched)
    from flair.data import Sentence
    from flair.models import SequenceTagger
    tagger = SequenceTagger.load(os.path.join(DIR_MODEL, 'best-model.pt'))
    sents = [Sentence(str(t)) for t in flat_txt]
    tagger.predict(sents, mini_batch_size=1024)
    out = []
    for s in sents:
        m = re.split(r'/', s.to_tagged_string())
        out.append(m[-1][:-1] if len(m) > 1 else 'X')
    return out, 'flair-fallback'

demo = '1013_Pid.B_2026_PN Sby.pdf'
with open(os.path.join(DIR_TXT, os.path.splitext(demo)[0] + '.txt'), encoding='utf-8') as f:
    c0 = f.read()
f0 = frags_dari_teks(c0)
pos0, cara0 = pos_untuk_dok(demo, f0)
print(f'[INFO] {demo}: {len(f0)} fragmen, POS via {cara0}')
k = 0
X0, flat0 = [], []
for fr in f0:
    plug = list(zip(fr, pos0[k:k+len(fr)])); k += len(fr)
    X0.append([token2features(plug, i) for i in range(len(plug))])
    flat0.extend(fr)
p0 = [l for s in CRF_DIAG.predict(X0) for l in s]
print('[INFO] distribusi label:', dict(Counter(p0).most_common(8)))
e0 = bio_to_entities(flat0, p0)
for t, vs in list(e0.items())[:5]:
    print(f'  {t} = {vs[0][:90]}')
assert any(l != 'O' for l in p0), 'MASIH ALL-O — jangan lanjut ke T1, laporkan!'
print('[OK] diagnostik hijau — silakan jalankan T1.')


[INFO] 1013_Pid.B_2026_PN Sby.pdf: 95 fragmen, POS via reuse-csv
[INFO] distribusi label: {'O': 4100, 'I_II': 83, 'I_BB': 81, 'I_TIMV': 65, 'B_TIMV': 32, 'I_VERN': 24, 'I_ARTV': 12, 'B_BB': 4}
  TIMV = 21 September 2026
  VERN = 1013 / Pid . B / 2026 / PN Sby tanggal 24 Juni
  BB = 1 ( buah Handphone merk Samsung Galaxy A17 warna hitam Imei 1 355500 imei 2 356772 ( dosbo
  PENA = tuntutan pidananya
  ARTV = Pasal 591 huruf a Undang-Undang Republik Indonesia Nomor 1 Tahun 2023 tentang KUHP
[OK] diagnostik hijau — silakan jalankan T1.


In [7]:
# T1 (Opsi A, ditulis ulang) — fitur inferensi IDENTIK M0 + POS reuse dari CSV.
# Tanpa Flair ulang kecuali fallback. Output: ekstraksi_putusan.csv (258 baris x 25 kolom).
# pip (sekali): pip install sklearn-crfsuite fuzzywuzzy python-Levenshtein
import joblib
from collections import Counter
from tqdm import tqdm

ENT_COLS = ['VERN','TIMV','JUDP','JUDG','REGI','ARTV','DEFN','PROS','CRIA','PENA','PUNI','BB',
            'PENGG','TERG','PEMOHON','KUASA','NIK','AMAR']
crf = joblib.load(os.path.join(DIR_MODEL, '600crf_surabaya.pkl'))
df_scan = pd.read_csv(os.path.join(DIR_CSV, 'data_all_surabaya.csv'))
dfml = pd.read_csv(os.path.join(DIR_CSV, 'data_mlformat_surabaya.csv')).fillna('')
posmap = {n: list(zip(g['token'].tolist(), g['pos'].tolist()))
          for n, g in dfml.groupby('nama_file')}

def token2features(frag, i):
    # SALINAN PERSIS M0 — jangan diubah agar skema fitur inferensi == training.
    token, postag = frag[i][0], frag[i][1]
    features = {'bias': 1.0, 'token': token, 'token.lower()': token.lower(),
                'token.isupper()': token.isupper(), 'token.istitle()': token.istitle(),
                'token.isdigit()': token.isdigit(), 'postag': postag}
    if i > 0:
        features.update({'prev1': frag[i-1][0], 'prev1.lower()': frag[i-1][0].lower(),
                         'prev1.isupper()': frag[i-1][0].isupper(), 'prev1.istitle()': frag[i-1][0].istitle(),
                         'prev1.isdigit()': frag[i-1][0].isdigit(), 'prev1.postag': frag[i-1][1]})
    else:
        features['BOF'] = True
    if i > 1:
        features.update({'prev2': frag[i-2][0], 'prev2.lower()': frag[i-2][0].lower(),
                         'prev2.isupper()': frag[i-2][0].isupper(), 'prev2.istitle()': frag[i-2][0].istitle(),
                         'prev2.isdigit()': frag[i-2][0].isdigit(), 'prev2.postag': frag[i-2][1]})
    if i < len(frag) - 1:
        features.update({'next1': frag[i+1][0], 'next1.lower()': frag[i+1][0].lower(),
                         'next1.isupper()': frag[i+1][0].isupper(), 'next1.istitle()': frag[i+1][0].istitle(),
                         'next1.isdigit()': frag[i+1][0].isdigit(), 'next1.postag': frag[i+1][1]})
    else:
        features['EOF'] = True
    if i < len(frag) - 2:
        features.update({'next2': frag[i+2][0], 'next2.lower()': frag[i+2][0].lower(),
                         'next2.isupper()': frag[i+2][0].isupper(), 'next2.istitle()': frag[i+2][0].istitle(),
                         'next2.isdigit()': frag[i+2][0].isdigit(), 'next2.postag': frag[i+2][1]})
    return features

def frags_dari_teks(content):
    # Tokenisasi IDENTIK representation_crf, tapi dikelompokkan per fragmen ;.
    frags = []
    for fragment in content.split(';'):
        fragment = re.sub(r'([\/\,\.():;])', r' \1 ', fragment)
        toks = re.findall(r'\S+', fragment)
        if toks:
            frags.append(toks)
    return frags

def pos_untuk_dok(nama_file, frags):
    ref = posmap.get(nama_file, [])
    flat_txt = [t for f in frags for t in f]
    if len(ref) == len(flat_txt) and all(a == b for (a, _), b in zip(ref, flat_txt)):
        return [p for _, p in ref], 'reuse-csv'
    from flair.data import Sentence
    from flair.models import SequenceTagger
    tagger = SequenceTagger.load(os.path.join(DIR_MODEL, 'best-model.pt'))
    sents = [Sentence(str(t)) for t in flat_txt]
    tagger.predict(sents, mini_batch_size=1024)
    out = []
    for s in sents:
        m = re.split(r'/', s.to_tagged_string())
        out.append(m[-1][:-1] if len(m) > 1 else 'X')
    return out, 'flair-fallback'

rows_out, cara_counter = [], Counter()
for _, r in tqdm(df_scan.iterrows(), total=len(df_scan), desc='Ekstraksi'):
    tp = os.path.join(DIR_TXT, os.path.splitext(r['nama_file'])[0] + '.txt')
    with open(tp, encoding='utf-8') as f:
        content = f.read()
    frags = frags_dari_teks(content)
    if not frags:
        rows_out.append({'nama_file': r['nama_file'], 'status_ekstraksi': 'gagal-kosong'})
        continue
    pos, cara = pos_untuk_dok(r['nama_file'], frags)
    cara_counter[cara] += 1
    Xd, flat = [], []
    k = 0
    for fr in frags:
        plug = list(zip(fr, pos[k:k+len(fr)])); k += len(fr)
        Xd.append([token2features(plug, i) for i in range(len(plug))])
        flat.extend(fr)
    pred = [l for s in crf.predict(Xd) for l in s]
    ent = bio_to_entities(flat, pred)
    wajib = ['VERN','TIMV','JUDP']
    terisi = sum(1 for kk in wajib if ent.get(kk))
    pihak = any(ent.get(kk) for kk in ['DEFN','PENGG','PEMOHON'])
    status = 'lengkap' if (terisi == 3 and pihak) else ('sebagian' if terisi >= 1 else 'gagal')
    row = {'nama_file': r['nama_file'], 'jenis_perkara': r['jenis_perkara'], 'tahun': r['tahun'],
           'pengadilan': r['pengadilan'], 'txt_path': tp, 'status_ekstraksi': status,
           'catatan': '' if status == 'lengkap' else f'wajib_terisi={terisi}/3 pihak={pihak}'}
    for kk in ENT_COLS:
        row[kk] = ' | '.join(ent.get(kk, []))[:2000]
    rows_out.append(row)
print('[INFO] sumber POS:', dict(cara_counter))
out_fin = os.path.join(DIR_CSV, 'ekstraksi_putusan.csv')
pd.DataFrame(rows_out).to_csv(out_fin, index=False)
print(f'[SAVE] {out_fin} ({len(rows_out)} baris = 1 PDF per baris, entitas = kolom)')
pd.DataFrame(rows_out)['status_ekstraksi'].value_counts()


Ekstraksi:  15%|█▍        | 38/258 [00:12<00:49,  4.41it/s]

2026-10-10 09:01:34,745 SequenceTagger predicts: Dictionary with 20 tags: O, PROPN, AUX, DET, NOUN, PRON, VERB, ADP, PUNCT, ADV, CCONJ, SCONJ, NUM, ADJ, PART, SYM, INTJ, X, <START>, <STOP>


Ekstraksi:  21%|██▏       | 55/258 [00:33<01:46,  1.91it/s]

2026-10-10 09:01:51,576 SequenceTagger predicts: Dictionary with 20 tags: O, PROPN, AUX, DET, NOUN, PRON, VERB, ADP, PUNCT, ADV, CCONJ, SCONJ, NUM, ADJ, PART, SYM, INTJ, X, <START>, <STOP>


Ekstraksi:  22%|██▏       | 58/258 [00:53<11:17,  3.39s/it]

...

Ekstraksi:  93%|█████████▎| 240/258 [05:00<00:47,  2.62s/it]

2026-10-10 09:06:18,608 SequenceTagger predicts: Dictionary with 20 tags: O, PROPN, AUX, DET, NOUN, PRON, VERB, ADP, PUNCT, ADV, CCONJ, SCONJ, NUM, ADJ, PART, SYM, INTJ, X, <START>, <STOP>


Ekstraksi: 100%|██████████| 258/258 [05:15<00:00,  1.22s/it]

[INFO] sumber POS: {'reuse-csv': 248, 'flair-fallback': 10}
[SAVE] /home/hujan/Documents/Kuliah/semester-7/Ekstraksi Informasi/Kode Tugas/MACHINE LEARNING IE/IE_project/output_csv/ekstraksi_putusan.csv (258 baris = 1 PDF per baris, entitas = kolom)


status_ekstraksi
sebagian    258
Name: count, dtype: int64

## App (opsional, cell 102–104 asli) — Gradio demo satu PDF

In [2]:
# App — demo Gradio 1 PDF (ditulis ulang, memakai fungsi T0+T1 di atas).
# Syarat: cell T0 dan T1 sudah di-run (namespace kernel). pip: pip install gradio
import gradio as gr

def predict_pdf(pdf_file):
    import tempfile
    src = pdf_file.name if hasattr(pdf_file, 'name') else pdf_file
    mentah = read_pdf_sby(src)
    content = clean_text_sby(mentah)
    frags = frags_dari_teks(content)
    if not frags:
        return 'Gagal membaca PDF.'
    Jaya = joblib.load(os.path.join(DIR_MODEL, '600crf_surabaya.pkl'))
    from flair.data import Sentence
    from flair.models import SequenceTagger
    tagger = SequenceTagger.load(os.path.join(DIR_MODEL, 'best-model.pt'))
    flat = [t for f in frags for t in f]
    sents = [Sentence(str(t)) for t in flat]
    tagger.predict(sents, mini_batch_size=1024)
    pos = []
    for s in sents:
        mm = re.split(r'/', s.to_tagged_string())
        pos.append(mm[-1][:-1] if len(mm) > 1 else 'X')
    Xd, k, flat2 = [], 0, []
    for fr in frags:
        plug = list(zip(fr, pos[k:k+len(fr)])); k += len(fr)
        Xd.append([token2features(plug, i) for i in range(len(plug))])
        flat2.extend(fr)
    pred = [l for s in Jaya.predict(Xd) for l in s]
    ent = bio_to_entities(flat2, pred)
    lines = []
    for kode, nama in KEY_MAPPING.items():
        vs = ent.get(kode, [])
        lines.append(f'{nama}: ' + (vs[0][:300] if vs else '-'))
    return '\n'.join(lines)

gr.Interface(fn=predict_pdf, inputs=gr.File(file_types=['.pdf']),
             outputs='text', title='Ekstraksi Putusan Surabaya').launch(share=True)


/home/hujan/miniconda3/envs/ekstraksi-informasi1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://45d78a15b32fda0df2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
